# Day 7 Exercise: The Lying Model
## Loan-default classification — completed analysis

**Objective:** demonstrate why accuracy can be misleading with imbalanced data, compare models on the same held-out test set, choose a probability threshold using an explicit cost assumption, and analyze missed defaulters.

**Reproducibility:** the dataset and train/test split use the exact settings supplied in the assignment. The data are synthetic. The generated feature values are not real borrower measurements or realistic naira amounts; interpret feature comparisons as patterns in this simulated dataset only.

**Recommendation in brief:** for a first-pass screening tool, use the logistic regression probabilities with a **0.20 threshold** as a provisional choice under the stated assumption that missing a defaulter is materially more costly than reviewing a good borrower. This is not a production lending decision: validate on real, representative data and assess fairness, calibration, and business costs before deployment.

## Part 1 — Build the loan book

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=2000, n_features=8, n_informative=4, n_redundant=1,
    weights=[0.92, 0.08], flip_y=0.03, class_sep=0.85, random_state=7
)

columns = ["income", "loan_amount", "age", "months_employed",
           "existing_loans", "credit_score", "debt_to_income", "late_payments_12m"]

df = pd.DataFrame(X, columns=columns)
df["defaulted"] = y

print("Dataset shape:", df.shape)
print(df["defaulted"].value_counts().rename(index={0: "repaid", 1: "default"}))
print("Default rate:", round(df["defaulted"].mean(), 3))
df.head()

Dataset shape: (2000, 9)
defaulted
repaid     1813
default     187
Name: count, dtype: int64
Default rate: 0.094


,income,loan_amount,age,months_employed,existing_loans,credit_score,debt_to_income,late_payments_12m,defaulted
0,-0.918795,0.264502,2.732221,-1.396712,-0.266967,0.392201,0.539481,1.578809,0
1,-1.841537,-1.758185,1.649915,1.628978,1.750556,-0.594751,1.272821,-1.561239,0
2,-1.376504,-1.393120,0.014917,-0.613674,-0.721855,-0.033645,-0.655524,0.186481,0
3,-0.520277,1.004231,0.567222,-1.078321,-0.830861,0.001173,0.355603,0.406391,0
4,-2.060806,-1.508931,0.386563,-0.860093,-0.503067,-1.106797,-0.390481,0.096481,0


## Train/test split — keep the test set unseen

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(len(X_train), "training rows |", len(X_test), "test rows")
print("Defaulters in test set:", int(y_test.sum()))

1500 training rows | 500 test rows
Defaulters in test set: 47


## Part 2 — Baseline that does nothing

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)

print("Dummy accuracy:", round(accuracy_score(y_test, dummy_preds), 3))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, dummy_preds))

Dummy accuracy: 0.906
Confusion matrix [[TN, FP], [FN, TP]]:
[[453   0]
 [ 47   0]]


**Interpretation:** the dummy model predicts “repaid” for every application. It gets 453 of 500 test cases right (90.6%) because repayment is the majority class, but it catches **0 of 47 defaulters**. This is the baseline every useful model must improve upon in business-relevant terms.

## Part 3 — Logistic regression

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, preds), 3))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds, target_names=["repaid", "default"], digits=2, zero_division=0))

Accuracy: 0.92
Confusion matrix [[TN, FP], [FN, TP]]:
[[451   2]
 [ 38   9]]
              precision    recall  f1-score   support

      repaid       0.92      1.00      0.96       453
     default       0.82      0.19      0.31        47

    accuracy                           0.92       500
   macro avg       0.87      0.59      0.63       500
weighted avg       0.91      0.92      0.90       500



### Written answer: What did the 1.4 percentage points buy the lender?

The model's accuracy increased from 90.6% to 92.0%, a gain of 1.4 percentage points. On this test set, however, it identified only 9 of 47 defaulters and missed 38. It also flagged two borrowers who would repay. The gain therefore buys only nine detected defaulters, while most defaults remain undetected. Whether that is worthwhile depends on loan value, recovery rates, and the cost of rejecting or reviewing good customers; accuracy alone cannot answer that.

## Challenge A — Make accuracy worse and the model better

In [5]:
balanced = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)
balanced.fit(X_train, y_train)
preds_b = balanced.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, preds_b), 3))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, preds_b))
print(classification_report(y_test, preds_b, target_names=["repaid", "default"], digits=2, zero_division=0))

Accuracy: 0.776
Confusion matrix [[TN, FP], [FN, TP]]:
[[358  95]
 [ 17  30]]
              precision    recall  f1-score   support

      repaid       0.95      0.79      0.86       453
     default       0.24      0.64      0.35        47

    accuracy                           0.78       500
   macro avg       0.60      0.71      0.61       500
weighted avg       0.89      0.78      0.82       500



### Written answer: Is the class-weighted model better or worse?

For a lender whose priority is finding potential defaults, the class-weighted model is better at the task of identifying defaulters, although it is not better on every metric. The confusion matrix is **[[358, 95], [17, 30]]**: it catches 30 of 47 defaulters and misses 17, compared with the ordinary logistic model's 9 caught and 38 missed. The trade-off is 95 good borrowers flagged as potential defaults and accuracy falling from 92.0% to 77.6%. The model is better only if the value of catching more defaulters outweighs the cost of reviewing or incorrectly rejecting good borrowers.

## Challenge B — Choose a probability threshold

In [6]:
from sklearn.metrics import precision_score, recall_score

probabilities = model.predict_proba(X_test)[:, 1]
threshold_rows = []
print(f"{'threshold':>10} {'recall':>8} {'precision':>10} {'flagged':>8} {'TN':>5} {'FP':>5} {'FN':>5} {'TP':>5}")
for t in [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]:
    preds_t = (probabilities >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds_t).ravel()
    row = {
        "threshold": t,
        "accuracy": accuracy_score(y_test, preds_t),
        "recall": recall_score(y_test, preds_t, zero_division=0),
        "precision": precision_score(y_test, preds_t, zero_division=0),
        "flagged": int(preds_t.sum()), "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }
    threshold_rows.append(row)
    print(f"{t:>10.2f} {row['recall']:>8.2f} {row['precision']:>10.2f} {row['flagged']:>8} {tn:>5} {fp:>5} {fn:>5} {tp:>5}")
threshold_table = pd.DataFrame(threshold_rows)
threshold_table

 threshold   recall  precision  flagged    TN    FP    FN    TP
      0.50     0.19       0.82       11   451     2    38     9
      0.40     0.26       0.67       18   447     6    35    12
      0.30     0.32       0.50       30   438    15    32    15
      0.20     0.45       0.45       47   427    26    26    21
      0.10     0.57       0.23      118   362    91    20    27
      0.05     0.70       0.17      195   291   162    14    33


,threshold,accuracy,recall,precision,flagged,TN,FP,FN,TP
0,0.50,0.920,0.191489,0.818182,11,451,2,38,9
1,0.40,0.918,0.255319,0.666667,18,447,6,35,12
2,0.30,0.906,0.319149,0.500000,30,438,15,32,15
3,0.20,0.896,0.446809,0.446809,47,427,26,26,21
4,0.10,0.778,0.574468,0.228814,118,362,91,20,27
5,0.05,0.648,0.702128,0.169231,195,291,162,14,33


### Threshold decision and explicit cost assumption

**Chosen threshold: 0.20.** Assume that a missed defaulter can cause a substantial loss of principal, while a false positive is sent for a low-cost manual review rather than automatically rejected. At 0.20, the model catches **21 of 47 defaulters (44.7% recall)**, compared with 9 at the default 0.50 threshold. It flags 47 applicants, of whom 21 are actual defaulters and 26 would repay (precision 44.7%). This accepts more review workload to reduce missed defaults.

This is a provisional operating point, not a universal optimum. If false positives lead to automatic rejection, or review is expensive, the threshold should be reconsidered using the lender's actual costs and customer-protection requirements.

## Challenge C — Catch a model that memorised

In [7]:
from sklearn.tree import DecisionTreeClassifier

deep = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
shallow = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)

print("Deep tree — train:", round(deep.score(X_train, y_train), 3),
      "| test:", round(deep.score(X_test, y_test), 3))
print("Depth-3 tree — train:", round(shallow.score(X_train, y_train), 3),
      "| test:", round(shallow.score(X_test, y_test), 3))
for name, tree in [("Deep tree", deep), ("Depth-3 tree", shallow)]:
    print("\n", name, "test confusion matrix:")
    print(confusion_matrix(y_test, tree.predict(X_test)))

Deep tree — train: 1.0 | test: 0.902
Depth-3 tree — train: 0.943 | test: 0.922

 Deep tree test confusion matrix:
[[435  18]
 [ 31  16]]

 Depth-3 tree test confusion matrix:
[[449   4]
 [ 35  12]]


### Written answer: Why does the shallow tree generalize better?

The unrestricted tree can keep splitting until it memorizes details and noise in the training examples. Its training accuracy is 100%, but test accuracy is only 90.2%, below the 90.6% dummy baseline. Limiting depth to three constrains complexity, so the tree cannot memorize as many individual cases. It has lower training accuracy (94.3%) but slightly higher test accuracy (92.2%). If only training accuracy had been reported, I would have selected the deep tree incorrectly. Test performance on unseen data is the more relevant check for generalization.

## Part 5 — Error analysis of the class-weighted model

In [8]:
test_df = pd.DataFrame(X_test, columns=columns)
test_df["actual"] = y_test
test_df["predicted"] = balanced.predict(X_test)
test_df["probability"] = balanced.predict_proba(X_test)[:, 1]

missed = test_df[(test_df["actual"] == 1) & (test_df["predicted"] == 0)]
caught = test_df[(test_df["actual"] == 1) & (test_df["predicted"] == 1)]

print(len(missed), "defaulters missed")
display(missed.sort_values("probability", ascending=False).head(10))
comparison = pd.DataFrame({
    "missed": missed[columns].mean(),
    "caught": caught[columns].mean()
})
comparison["difference (missed - caught)"] = comparison["missed"] - comparison["caught"]
display(comparison.round(2))
print("Missed-defaulter predicted probability summary:")
display(missed["probability"].describe().round(3))

17 defaulters missed


,income,loan_amount,age,months_employed,existing_loans,credit_score,debt_to_income,late_payments_12m,actual,predicted,probability
335,2.044454,0.209550,0.069818,-0.326909,0.350912,2.241093,-0.672653,1.818542,1,0,0.468663
116,-0.269323,1.299351,-0.288072,-3.399208,2.028629,0.829835,-0.220931,2.396438,1,0,0.417951
433,0.068840,1.376970,-0.016365,-3.587908,-1.686218,0.254494,1.259591,2.908166,1,0,0.398309
486,-0.578013,0.097075,0.598651,-0.214851,-1.057215,-0.123337,-1.246026,-0.050068,1,0,0.380391
58,1.764601,0.477137,0.815408,-1.811411,-1.101770,-3.042637,-0.065008,3.211774,1,0,0.297800
248,-0.410474,1.099338,1.552991,0.810463,-1.823105,-0.227796,1.200417,-1.015199,1,0,0.282224
179,-0.346579,-1.508905,0.417276,0.028676,1.262860,-0.251460,0.968702,0.558854,1,0,0.266819
167,-1.874132,-1.277142,0.814503,-0.326278,-0.533810,-1.770321,0.289239,-0.222782,1,0,0.219394
497,-0.735079,-0.551122,1.498643,1.721922,1.649005,-0.913274,0.083547,-1.414639,1,0,0.204099
323,-0.364610,-0.370659,0.885062,-1.269634,0.562126,-0.680910,-1.421801,1.465489,1,0,0.190094


,missed,caught,difference (missed - caught)
income,-0.19,-0.01,-0.18
loan_amount,0.13,-0.03,0.16
age,1.04,-1.08,2.12
months_employed,-0.76,0.06,-0.83
existing_loans,0.14,-0.01,0.15
credit_score,-0.10,0.23,-0.33
debt_to_income,-0.11,0.18,-0.29
late_payments_12m,0.93,-0.47,1.41


Missed-defaulter predicted probability summary:


count    17.000
mean      0.233
std       0.125
min       0.073
25%       0.131
50%       0.204
75%       0.298
max       0.469
Name: probability, dtype: float64

### Error analysis findings

The class-weighted model misses **17 of the 47 actual defaulters** in the test set, so its default recall is **30/47 = 63.8%**. For these 17 missed cases, the model's predicted default probability has a **mean of 0.233, median of 0.204, interquartile range of 0.131–0.298, and full range of 0.073–0.469**. Four of the 17 missed cases have scores between **0.35 and 0.50** (the four highest are approximately 0.380–0.469), meaning their scores are relatively close to the model's 0.50 classification cutoff. The other 13 have scores below 0.35; therefore, lowering the cutoff slightly might recover some near-cutoff cases, but would not by itself explain or eliminate every miss. Any threshold change would also increase the number of non-defaulters flagged, so its value must be checked against the resulting confusion matrix and review cost.

Comparing feature averages for missed versus caught defaulters shows the largest differences in this sample for **`age` (1.04 vs. -1.08; difference +2.12)** and **`late_payments_12m` (0.93 vs. -0.47; difference +1.41)**. Missed cases also have a lower average **`months_employed` (-0.76 vs. 0.06; difference -0.83)**. The remaining mean differences are smaller: `credit_score` is -0.10 vs. 0.23, `debt_to_income` is -0.11 vs. 0.18, `income` is -0.19 vs. -0.01, `loan_amount` is 0.13 vs. -0.03, and `existing_loans` is 0.14 vs. -0.01.

These averages are **descriptive, not causal**. They do not prove that the model is systematically blind to age, late payments, or employment history, and the feature values are synthetic rather than real-world borrower units. To test whether the errors reflect a repeatable weakness, I would inspect the individual missed cases, compare error rates across feature ranges, and repeat the analysis on representative real loan data. I would also evaluate alternative thresholds on validation data and report both the additional defaults caught and the extra good borrowers flagged before recommending a policy change.

## Part 6 — Results table (same 500-case test set)

In [9]:
def summarize(name, estimator, predictions):
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Default recall": recall_score(y_test, predictions, zero_division=0),
        "Default precision": precision_score(y_test, predictions, zero_division=0),
        "Defaulters caught": f"{tp} of {int(y_test.sum())}",
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

threshold = 0.20
threshold_preds = (probabilities >= threshold).astype(int)
results = pd.DataFrame([
    summarize("Dummy (predicts repaid)", dummy, dummy_preds),
    summarize("Logistic regression", model, preds),
    summarize("Logistic, class-weighted", balanced, preds_b),
    summarize("Logistic, threshold 0.20", model, threshold_preds),
    summarize("Decision tree, depth 3", shallow, shallow.predict(X_test))
])
display(results.style.format({
    "Accuracy": "{:.3f}", "Default recall": "{:.3f}", "Default precision": "{:.3f}"
}))

,Model,Accuracy,Default recall,Default precision,Defaulters caught,TN,FP,FN,TP
0,Dummy (predicts repaid),0.906,0.000,0.000,0 of 47,453,0,47,0
1,Logistic regression,0.920,0.191,0.818,9 of 47,451,2,38,9
2,"Logistic, class-weighted",0.776,0.638,0.240,30 of 47,358,95,17,30
3,"Logistic, threshold 0.20",0.896,0.447,0.447,21 of 47,427,26,26,21
4,"Decision tree, depth 3",0.922,0.255,0.750,12 of 47,449,4,35,12


### Recommendation

For a first-pass **human-review screening** workflow, I would pilot logistic regression at threshold **0.20**, because it catches 21 of 47 test-set defaulters rather than 9 at threshold 0.50, while making the review workload explicit (47 applicants flagged). The class-weighted model catches more defaulters (30) but flags 125 applicants, including 95 who repaid; it may be preferable if missed defaults are much more costly and review capacity is sufficient.

I would not deploy this synthetic-data model directly to make lending decisions. Before deployment, I would require real historical data, a time-based/out-of-sample validation, calibration and stability checks, fairness and regulatory review, an agreed cost matrix, and human oversight. I would change the recommendation if real-world costs, recall/precision, or fairness tests showed a different operating point was safer and more valuable.

## Bonus — Cross-validation and a different model

In [10]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

cv_recall = cross_val_score(balanced, X, y, cv=5, scoring="recall")
print("Class-weighted logistic regression, 5-fold recall:")
print(np.round(cv_recall, 3))
print("Mean recall:", round(cv_recall.mean(), 3), "| range:", round(cv_recall.min(), 3), "to", round(cv_recall.max(), 3))

forest = RandomForestClassifier(class_weight="balanced", random_state=7, n_estimators=200)
forest.fit(X_train, y_train)
forest_preds = forest.predict(X_test)
print("\nRandom forest — accuracy:", round(accuracy_score(y_test, forest_preds), 3))
print("Default precision:", round(precision_score(y_test, forest_preds, zero_division=0), 3))
print("Default recall:", round(recall_score(y_test, forest_preds, zero_division=0), 3))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, forest_preds))

Class-weighted logistic regression, 5-fold recall:
[0.73  0.676 0.784 0.789 0.868]
Mean recall: 0.769 | range: 0.676 to 0.868



Random forest — accuracy: 0.926
Default precision: 0.812
Default recall: 0.277
Confusion matrix [[TN, FP], [FN, TP]]:
[[450   3]
 [ 34  13]]


**Bonus interpretation:** cross-validation recall varies across folds, which is a reminder that a single test split is uncertain. The random forest is an additional comparison, not automatically a better choice; judge it using recall, precision, false-positive workload, and repeatability rather than accuracy alone.

## Five-minute presentation — speaking notes

1. **The trap (about 1 minute):** “The dataset has only about 9% defaults. A model that predicts repayment for everyone scores 90.6% accuracy. Logistic regression reaches 92%, but catches only 9 of 47 defaulters and misses 38. So the headline accuracy hides the actual business failure.”
2. **The fix (about 2 minutes):** “Class weighting lowers accuracy to 77.6%, but catches 30 of 47 defaults. Alternatively, lowering the standard logistic model's threshold to 0.20 catches 21 of 47, with 26 false positives among 47 flagged cases. I recommend piloting that threshold for manual review under the assumption that missing a defaulter is more costly than reviewing a good borrower.”
3. **The honest limitation (about 1 minute):** “The unrestricted tree memorizes training data: 100% training accuracy but 90.2% test accuracy. The depth-3 tree generalizes better. Our dataset is synthetic, and the selected threshold depends on costs we have not measured. Real loan data, cost estimates, fairness checks, and monitoring are required before deployment.”
4. **If asked, ‘Why should the lender trust this?’** “They should not trust it blindly. They should trust the transparent test-set comparison as an initial experiment, then require independent validation, documented costs, human review, and ongoing monitoring before any operational use.”